# Cardboard Beverage Carton LCA

Cradle-to-grave carbon footprint of one 0.5 L paperboard beverage carton, Brightway for the calculation, BAFU for the background. 

## 1. Project setup

In [ ]:
import brightway2 as bw

bw.projects.set_current("Cardboard Carton LCA")
bw.bw2setup() 

print(list(bw.databases))

Creating default biosphere

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds
Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:54:31
  Finished: 08/06/2026 11:54:32
  Total time elapsed: 00:00:00
  CPU %: 76.40
  Memory %: 2.49
Created database: biosphere3
Creating default LCIA methods

Wrote 762 LCIA methods with 227223 characterization factors
Creating core data migrations

['biosphere3']


## 2. Import BAFU (ecoSpold1)

Same procedure as the other notebooks. Update the path below to the extracted ecoSpold1 folder.

In [ ]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files" 

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 25.56 seconds


In [3]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

In [4]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 22.93 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

In [5]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


In [6]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

Check what's still unlinked before dropping it, same as before.

In [7]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

In [8]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.69 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [9]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:56:15
  Finished: 08/06/2026 11:56:32
  Total time elapsed: 00:00:17
  CPU %: 86.70
  Memory %: 0.81
Created database: BAFU
['biosphere3', 'BAFU']


## 3. Search BAFU for paperboard production and the PE coating

Run each search and look at the real results before picking anything. Board and packaging materials showed up a lot in earlier notebooks' searches (corrugated board, packaging cardboard disposal), so I'd expect reasonable coverage here.

In [10]:
ei = bw.Database("BAFU")

# Paperboard or liquid packaging board specifically
for r in ei.search("paperboard"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("liquid packaging board"):
    print(r, "|", r["location"])

'Disposal, packaging cardboard, 19.6% water, to construction waste landfill' (kilogram, CH, ['waste management', 'construction waste landfill']) | CH
'Disposal, packaging cardboard, 19.6% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
---
'Liquid packaging board, at plant' (kilogram, RER, ['paper+ board', 'board']) | RER
'Production of liquid packaging board containers, at plant' (kilogram, RER, ['paper+ board', 'packagings']) | RER


In [11]:
# Broader board and paper search, in case the exact term above doesn't match
for r in ei.search("cardboard"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("kraft paper"):
    print(r, "|", r["location"])

'xx Recycling cardboard' (kilogram, RER, ['waste treatment, obsolete', 'recycling, obsolete']) | RER
'Disposal, building, plaster-cardboard sandwich, to recycling' (kilogram, CH, ['construction waste']) | CH
'Disposal, packaging cardboard, 19.6% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
'Disposal, steel board, 10 mm, powder coated, cardboard filled' (square meter, CH, ['construction', 'kitchen']) | CH
'Disposal, packaging cardboard, 19.6% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill']) | CH
'Steel board, 20 mm, powder coated, cardboard filled, at plant' (square meter, CH, ['construction', 'kitchen']) | CH
'xx Disposal, building, plaster-cardboard sandwich, to final disposal' (kilogram, CH, ['waste treatment, obsolete', 'construction waste, obsolete']) | CH
'xx Disposal, building, plaster-cardboard sandwich, to sorting plant' (kilogram, CH, ['waste treatment, obsolete', 'construction waste, obsole

In [12]:
# The thin polyethylene coating that makes the board liquid-proof.
# The PET bottle notebook already confirmed a low-density polyethylene
# disposal activity exists in BAFU, so LDPE production is a reasonable bet too
for r in ei.search("polyethylene, low density"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("LDPE"):
    print(r, "|", r["location"])

---
'Polyethylene, LDPE, granulate, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'Packaging film, LDPE, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'Cable, network cable, category 5, without plugs, at plant' (meter, GLO, ['electronics', 'devices']) | GLO
'Kitchen worktop, chromium steel, high-end, at plant' (square meter, CH, ['construction', 'kitchen']) | CH
'Plugs, inlet and outlet, for printer cable, at plant' (unit, GLO, ['electronics', 'devices']) | GLO


Fill in the picks below once the searches above have been run and the correct index is known for each.

In [13]:
# Replace the "..." search terms and [0] indices with the confirmed picks
# paperboard = ei.search("...")[0]
# pe_coating = ei.search("...")[0]

Transport reuses the confirmed pick from the other notebooks. For end-of-life, `'Disposal, packaging cardboard, 19.6% water, to sanitary landfill'` was seen directly in the `sanitary landfill` search results from the PET bottle and aluminium can notebooks, at roughly this position, check the printed name before trusting the index.

In [14]:
truck = ei.search("transport lorry")[2]
print(truck)  # expect: 'Transport, freight, lorry, fleet average' (ton kilometer, CH)

landfill = ei.search("sanitary landfill")[17]
print(landfill)  # expect: 'Disposal, packaging cardboard, 19.6% water, to sanitary landfill' (CH)

'Transport, freight, lorry, fleet average' (ton kilometer, RER, ['transport systems', 'road\\Transformation'])
'Disposal, plastics, mixture, 15.3% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill'])


## 4. Material split and end-of-life assumptions

A typical fresh beverage carton is roughly 80 to 90 percent paperboard by weight, with the rest being the thin PE coating. I'm using 85 percent board, 15 percent PE here, adjust if a specific carton spec is available. Paper-based packaging has good collection infrastructure in most of Europe, so recycling tends to dominate end-of-life, though the PE coating usually can't be recovered in standard paper recycling and ends up incinerated or landfilled along with residual fibre losses.

In [15]:
# ---- Physical parameters ----
CARTON_MASS_KG = 0.028   # a 0.5 L paperboard beverage carton weighs about 28 g
BOARD_SHARE = 0.85        # share of carton mass that is paperboard
PE_SHARE = 0.15            # share that is the PE liquid-proofing coating
DIST_KM = 100               # factory to retailer distance, same assumption as the other notebooks

# ---- End-of-life split ----
EoL_RECYCLE = 0.70    # paper recycling collection is generally well established
EoL_LANDFILL = 0.30

print(f"Board mass: {CARTON_MASS_KG * BOARD_SHARE:.4f} kg")
print(f"PE mass:    {CARTON_MASS_KG * PE_SHARE:.4f} kg")

Board mass: 0.0238 kg
PE mass:    0.0042 kg


## 5. Build the foreground activity

Same simplified-until-confirmed pattern as the other notebooks. There's no confirmed BAFU cardboard recycling activity yet, so for now the recycled share is routed through the same landfill pick as a placeholder, uncomment and adjust once a proper recycling activity is found.

In [16]:
DB_NAME = "Cardboard Carton"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

exchanges = [
    {"input": (DB_NAME, "cardboard_carton_0.5L"), "amount": 1, "unit": "piece", "type": "production"},

    # ---- Material stage ----
    # {"input": paperboard.key, "amount": CARTON_MASS_KG * BOARD_SHARE, "unit": "kilogram", "type": "technosphere"},
    # {"input": pe_coating.key, "amount": CARTON_MASS_KG * PE_SHARE, "unit": "kilogram", "type": "technosphere"},

    # ---- Distribution ----
    {"input": truck.key, "amount": CARTON_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},

    # ---- End of life ----
    {"input": landfill.key, "amount": CARTON_MASS_KG * (EoL_RECYCLE + EoL_LANDFILL), "unit": "kilogram", "type": "technosphere"},
]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "cardboard_carton_0.5L"): {
        "name": "Cardboard beverage carton, 0.5 litre, cradle-to-grave",
        "reference product": "Cardboard beverage carton, 0.5 litre, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": exchanges,
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:57:06
  Finished: 08/06/2026 11:57:06
  Total time elapsed: 00:00:00
  CPU %: 60.70
  Memory %: 1.93
Database written: ['biosphere3', 'BAFU', 'Cardboard Carton']


Note to self: until the material picks are uncommented, this only accounts for transport and end-of-life. The score in section 6 will be too low until the paperboard and PE coating are filled in.

## 6. Run the LCA

Same method as the other notebooks.

In [17]:
carton = bw.get_activity((DB_NAME, "cardboard_carton_0.5L"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({carton: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per carton")

GWP 100a: 0.0034 kg CO2-eq per carton


## 8. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number.

Two things to know first: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment and restart the kernel. And Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data, the foreground exchanges written here don't have any set by default, so they'll stay fixed across iterations unless it's added explicitly.

In [18]:
from bw2calc import MonteCarloLCA

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = MonteCarloLCA({carton: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per carton")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per carton")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   0.0040 kg CO2-eq per carton
Median: 0.0035 kg CO2-eq per carton
Std dev: 0.0018
5th to 95th percentile: 0.0021 to 0.0073 kg CO2-eq
